# 🪰 Musca domestica Cohort Simulation — Colab Launcher

This notebook clones the simulation from GitHub, installs dependencies, and
opens a public link so you can use the full Streamlit interface from inside
Google Colab.

**No account, no token, no sign-up required.**

**How to use:**
1. Run **Cell 1** once to install packages and download the tunnel tool (~60 s)
2. Run **Cell 2** — a public URL will appear; click it to open the app
3. When finished, run **Cell 3** to shut everything down cleanly

> The public URL is provided by
> [Cloudflare Tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/)
> and stays active for the duration of your Colab session.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Clone repository, install Python packages, download cloudflared
# Run once per Colab session.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os

GITHUB_REPO = 'https://github.com/ackuyucu/Ekoloji_Lab.git'
REPO_DIR    = 'Ekoloji_Lab'
APP_PATH    = os.path.join(REPO_DIR, 'musca_streamlit_app', 'app.py')

# ── Clone (or pull if already present) ───────────────────────────────────────
if not os.path.isdir(REPO_DIR):
    print('Cloning repository…')
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, REPO_DIR],
                   check=True)
else:
    print('Repository already present — pulling latest…')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# ── Install Python dependencies ───────────────────────────────────────────────
print('Installing Python packages…')
req_file = os.path.join(REPO_DIR, 'musca_streamlit_app', 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', req_file], check=True)

# ── Download cloudflared (Cloudflare Tunnel — no account needed) ──────────────
if not os.path.isfile('/tmp/cloudflared'):
    print('Downloading cloudflared…')
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/tmp/cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', '/tmp/cloudflared'], check=True)

print('\n✅  Ready.  Proceed to Cell 2.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Launch the app and open a public link
# After running this cell, click the URL that appears to open the app.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, time, re, os
from IPython.display import display, HTML

PORT     = 8501
ST_LOG   = '/tmp/streamlit.log'
CF_LOG   = '/tmp/cloudflared.log'

# Kill any leftover processes from a previous run
subprocess.run(['pkill', '-f', 'streamlit'],    capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'],  capture_output=True)
time.sleep(1)

# ── Start Streamlit in the background ─────────────────────────────────────────
with open(ST_LOG, 'w') as log:
    subprocess.Popen(
        [
            'streamlit', 'run', APP_PATH,
            f'--server.port={PORT}',
            '--server.headless=true',
            '--server.enableCORS=false',
            '--server.enableXsrfProtection=false',
        ],
        stdout=log, stderr=log
    )

# Wait until Streamlit is ready
print('Starting Streamlit', end='', flush=True)
for _ in range(30):
    time.sleep(1)
    with open(ST_LOG) as f:
        if 'You can now view' in f.read():
            break
    print('.', end='', flush=True)
print()

# ── Start Cloudflare Tunnel ────────────────────────────────────────────────────
with open(CF_LOG, 'w') as log:
    subprocess.Popen(
        ['/tmp/cloudflared', 'tunnel', '--url', f'localhost:{PORT}'],
        stdout=log, stderr=log
    )

# Wait for the public URL to appear in the log
public_url = None
print('Opening tunnel', end='', flush=True)
for _ in range(30):
    time.sleep(1)
    with open(CF_LOG) as f:
        content = f.read()
    match = re.search(r'https://[\w\-]+\.trycloudflare\.com', content)
    if match:
        public_url = match.group(0)
        break
    print('.', end='', flush=True)
print()

if public_url:
    display(HTML(f"""
    <div style="background:#f0fdf4; border:2px solid #16a34a; border-radius:10px;
                padding:1rem 1.5rem; font-family:sans-serif; margin-top:0.5rem">
      <h3 style="color:#15803d; margin:0 0 0.4rem 0">✅  App is running!</h3>
      <p style="margin:0; font-size:1.05rem">
        Click to open:
        <a href="{public_url}" target="_blank"
           style="color:#1d4ed8; font-weight:700">{public_url}</a>
      </p>
      <p style="margin:0.5rem 0 0 0; color:#555; font-size:0.85rem">
        The link stays active as long as this Colab session is running.
        No account or token required.
      </p>
    </div>
    """))
else:
    print('⚠️  Could not find the tunnel URL.  Check /tmp/cloudflared.log for details.')
    with open(CF_LOG) as f:
        print(f.read()[-1500:])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — (Optional) Shut down the app and close the tunnel
# ─────────────────────────────────────────────────────────────────────────────
import subprocess

subprocess.run(['pkill', '-f', 'streamlit'],   capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
print('🛑  Streamlit stopped and tunnel closed.')